# CerberusVision Phase 4 Temiz QLoRA Eğitimi

Bu notebook Google Colab Pro A100 üzerinde Qwen2.5-7B-Instruct modelini temiz ve hash doğrulamalı Phase 3 splitleriyle eğitir.

Eski LoRA adaptörü veya eski checkpoint kullanılmaz. Eğitim sabitlenmiş temel model revision'ından yeni bir adaptörle başlar. Validation seti model seçimi ve early stopping için kullanılır. Bağımsız holdout bu notebook'a yüklenmez.

## 1. Sabitlenmiş eğitim ortamı

Colab çalışma zamanı türünü A100 GPU olarak seçin. Aşağıdaki hücre yalnız PyPI üzerindeki sabitlenmiş paket sürümlerini kurar.

In [ ]:
%pip install -q --upgrade transformers==5.14.1 trl==1.8.0 peft==0.19.1 accelerate==1.14.0 datasets==5.0.0 bitsandbytes==0.49.2 safetensors

## 2. Drive, A100 ve deney sabitleri

İlk eğitimde `RESUME_FROM_MATCHING_CHECKPOINT` değeri `False` kalmalıdır. Yalnız aynı Phase 4 koşusu Colab kesintisi nedeniyle yarıda kaldıysa `True` yapılabilir.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import os
import random
import re
import shutil
import statistics
import torch
import unicodedata

drive.mount('/content/drive')

SEED = 3407
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
BASE_MODEL_REVISION = 'a09a35458c702b33eeacc393d103063234e8bc28'
PHASE4_CONTRACT_SHA256 = 'b6d941b577881324d4dc5275681d90aced27deca6215055190c49f1ef6afd0ee'
MAX_SEQUENCE_LENGTH = 2048
PACKAGE_DIR = Path('/content/drive/MyDrive/CerberusVision_Colab_Egitim_Seti')
RESUME_FROM_MATCHING_CHECKPOINT = False

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU bulunamadi')

GPU_NAME = torch.cuda.get_device_name(0)
if 'A100' not in GPU_NAME.upper():
    raise RuntimeError(f'A100 bekleniyordu, bulunan GPU: {GPU_NAME}')
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
if GPU_MEMORY_GB < 39:
    raise RuntimeError(f'En az 40 GB A100 bellegi bekleniyordu: {GPU_MEMORY_GB:.2f} GB')

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(json.dumps({'gpu': GPU_NAME, 'gpu_memory_gb': round(GPU_MEMORY_GB, 2), 'seed': SEED, 'package_dir': str(PACKAGE_DIR)}, ensure_ascii=False, indent=2))

## 3. Veri, manifest ve sızıntı doğrulaması

Bu hücre train, validation, split manifesti ve Phase 4 sözleşmesinin SHA-256 değerlerini doğrular. Ayrıca JSON çıktıları, zorunlu alanları ve train/validation exact normalize metin ayrımını yeniden kontrol eder.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def normalized_input_hash(value):
    normalized = unicodedata.normalize('NFKC', value).casefold()
    normalized = re.sub(r'[^\w]+', ' ', normalized, flags=re.UNICODE)
    normalized = ' '.join(normalized.split())
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

required_files = ['train.jsonl', 'validation.jsonl', 'manifest.json', 'phase4_contract.json']
missing_files = [name for name in required_files if not (PACKAGE_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Eksik Phase 4 dosyalari: {missing_files}')

contract_path = PACKAGE_DIR / 'phase4_contract.json'
contract_actual_hash = sha256_file(contract_path)
if contract_actual_hash != PHASE4_CONTRACT_SHA256:
    raise RuntimeError(f'Phase 4 sozlesme hash uyusmazligi: {contract_actual_hash}')
contract = json.loads(contract_path.read_text(encoding='utf-8'))
split_manifest = json.loads((PACKAGE_DIR / 'manifest.json').read_text(encoding='utf-8'))
dataset_contract = contract['dataset']

expected_hashes = {
    'train.jsonl': dataset_contract['train_sha256'],
    'validation.jsonl': dataset_contract['validation_sha256'],
    'manifest.json': dataset_contract['split_manifest_sha256'],
}
actual_hashes = {name: sha256_file(PACKAGE_DIR / name) for name in expected_hashes}
hash_mismatches = {name: {'expected': expected_hashes[name], 'actual': actual_hashes[name]} for name in expected_hashes if expected_hashes[name] != actual_hashes[name]}
if hash_mismatches:
    raise RuntimeError(f'Phase 4 hash uyusmazligi: {hash_mismatches}')

if split_manifest['files']['train.jsonl'] != actual_hashes['train.jsonl']:
    raise RuntimeError('Split manifest train hash uyusmazligi')
if split_manifest['files']['validation.jsonl'] != actual_hashes['validation.jsonl']:
    raise RuntimeError('Split manifest validation hash uyusmazligi')

train_rows = load_jsonl(PACKAGE_DIR / 'train.jsonl')
validation_rows = load_jsonl(PACKAGE_DIR / 'validation.jsonl')
if len(train_rows) != dataset_contract['train_records']:
    raise RuntimeError('Train kayit sayisi uyusmuyor')
if len(validation_rows) != dataset_contract['validation_records']:
    raise RuntimeError('Validation kayit sayisi uyusmuyor')

required_columns = {'instructions', 'input', 'output'}
for split_name, rows in [('train', train_rows), ('validation', validation_rows)]:
    for row_index, row in enumerate(rows):
        if not required_columns.issubset(row):
            raise RuntimeError(f'{split_name} kaydi eksik alan iceriyor: {row_index}')
        json.loads(row['output'])

train_input_hashes = {normalized_input_hash(row['input']) for row in train_rows}
validation_input_hashes = {normalized_input_hash(row['input']) for row in validation_rows}
split_overlap = train_input_hashes & validation_input_hashes
if split_overlap:
    raise RuntimeError(f'Train validation exact sizintisi bulundu: {len(split_overlap)}')

print(json.dumps({'train_records': len(train_rows), 'validation_records': len(validation_rows), 'split_overlap': len(split_overlap), 'hashes': actual_hashes}, ensure_ascii=False, indent=2))

## 4. Güvenli tokenizer, prompt formatı ve token uzunluğu denetimi

Tokenizer sabit model revision'ından `trust_remote_code` olmadan yüklenir. Veri prompt-completion biçimine çevrilir. Maksimum uzunluğu aşan tek bir örnek bile varsa eğitim durur; sessiz truncation yapılmaz.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

SYSTEM_PROMPT = 'You are a shipping instruction document parser. Return only valid JSON matching the requested structure. Use only evidence present in the OCR text. Set absent fields to null and never fabricate values.'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def row_messages(row):
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': row['instructions'] + '\n\n' + row['input']},
        {'role': 'assistant', 'content': row['output']},
    ]

def row_prompt_completion(row):
    messages = row_messages(row)
    return {'prompt': messages[:2], 'completion': [messages[2]]}

def token_length(row):
    rendered = tokenizer.apply_chat_template(row_messages(row), tokenize=False, add_generation_prompt=False)
    return len(tokenizer(rendered, add_special_tokens=False)['input_ids'])

train_token_lengths = [token_length(row) for row in train_rows]
validation_token_lengths = [token_length(row) for row in validation_rows]
all_token_lengths = train_token_lengths + validation_token_lengths
over_limit = [length for length in all_token_lengths if length > MAX_SEQUENCE_LENGTH]
if over_limit:
    raise RuntimeError(f'Truncation riski: {len(over_limit)} kayit {MAX_SEQUENCE_LENGTH} token sinirini asiyor')

train_dataset = Dataset.from_list([row_prompt_completion(row) for row in train_rows])
validation_dataset = Dataset.from_list([row_prompt_completion(row) for row in validation_rows])

length_report = {
    'minimum': min(all_token_lengths),
    'median': statistics.median(all_token_lengths),
    'p95': sorted(all_token_lengths)[math.floor((len(all_token_lengths) - 1) * 0.95)],
    'maximum': max(all_token_lengths),
    'limit': MAX_SEQUENCE_LENGTH,
    'truncated_records': len(over_limit),
}
print(json.dumps(length_report, ensure_ascii=False, indent=2))

## 5. Deney sözleşmesi ve izole checkpoint dizini

Veri, model revision'ı ve hiperparametrelerden deney hash'i üretilir. Checkpoint yalnız aynı sözleşmeyle devam ettirilebilir. Eski eğitim dizinleri bu akışta aranmaz.

In [ ]:
installed_packages = {name: importlib.metadata.version(name) for name in contract['packages']}
package_mismatches = {name: {'expected': contract['packages'][name], 'actual': installed_packages[name]} for name in contract['packages'] if contract['packages'][name] != installed_packages[name]}
if package_mismatches:
    raise RuntimeError(f'Paket surumu uyusmazligi: {package_mismatches}')

runtime_contract = {
    'phase4_contract': contract,
    'gpu': GPU_NAME,
    'gpu_memory_gb': GPU_MEMORY_GB,
    'torch': torch.__version__,
    'packages': installed_packages,
    'token_lengths': length_report,
}
runtime_contract_json = json.dumps(runtime_contract, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
runtime_contract_hash = hashlib.sha256(runtime_contract_json.encode('utf-8')).hexdigest()
RUN_ROOT = Path('/content/drive/MyDrive/CerberusVision/phase4_runs') / f"{contract['run_name']}_{runtime_contract_hash[:12]}"
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
ADAPTER_DIR = RUN_ROOT / 'adapter_best'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

runtime_contract_path = RUN_ROOT / 'runtime_contract.json'
if runtime_contract_path.exists():
    existing_contract = json.loads(runtime_contract_path.read_text(encoding='utf-8'))
    existing_json = json.dumps(existing_contract, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    if hashlib.sha256(existing_json.encode('utf-8')).hexdigest() != runtime_contract_hash:
        raise RuntimeError('Checkpoint dizini farkli bir deney sozlesmesine ait')
else:
    runtime_contract_path.write_text(json.dumps(runtime_contract, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps({'run_root': str(RUN_ROOT), 'contract_hash': runtime_contract_hash, 'resume_enabled': RESUME_FROM_MATCHING_CHECKPOINT}, ensure_ascii=False, indent=2))

## 6. Sabit temel modelden yeni QLoRA adaptörü

Model NF4 4-bit olarak yüklenir. QLoRA tüm temel projeksiyon katmanlarına uygulanır. Eski adaptör yüklenmez ve remote model kodu çalıştırılmaz.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_MODEL_REVISION,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map={'': 0},
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Trainer, validation ve early stopping

En fazla 10 epoch çalışır. Validation loss her 10 optimizer adımında ölçülür. Üç değerlendirme boyunca anlamlı iyileşme yoksa durur ve en iyi checkpoint'i geri yükler. Loss yalnız assistant completion tokenleri üzerinden hesaplanır.

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

training_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=9,
    weight_decay=0.01,
    max_grad_norm=0.3,
    lr_scheduler_type='linear',
    optim='paged_adamw_8bit',
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy='steps',
    eval_steps=10,
    save_strategy='steps',
    save_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_strategy='steps',
    logging_steps=1,
    logging_first_step=True,
    seed=SEED,
    data_seed=SEED,
    report_to='none',
    max_length=MAX_SEQUENCE_LENGTH,
    packing=False,
    completion_only_loss=True,
    dataset_num_proc=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.0005)],
)

baseline_metrics = trainer.evaluate(metric_key_prefix='baseline')
print(json.dumps(baseline_metrics, ensure_ascii=False, indent=2))

## 8. Eğitimi başlat

İlk koşuda yeni adaptör oluşturulur. Resume yalnız bu notebook'un aynı sözleşme hash'li checkpoint dizininden yapılabilir.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR)) if CHECKPOINT_DIR.exists() else None
if RESUME_FROM_MATCHING_CHECKPOINT and last_checkpoint is None:
    raise RuntimeError('Resume istendi ancak uyumlu Phase 4 checkpoint bulunamadi')
resume_checkpoint = last_checkpoint if RESUME_FROM_MATCHING_CHECKPOINT else None

training_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
final_metrics = trainer.evaluate(metric_key_prefix='final')
print(json.dumps({'train': training_result.metrics, 'final': final_metrics, 'best_checkpoint': trainer.state.best_model_checkpoint, 'best_metric': trainer.state.best_metric}, ensure_ascii=False, indent=2, default=str))

## 9. En iyi adaptörü ve deney raporunu kaydet

Kaydedilen model son epoch değil, `load_best_model_at_end` tarafından geri yüklenen en düşük validation loss checkpoint'idir.

In [ ]:
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

training_report = {
    'runtime_contract_hash': runtime_contract_hash,
    'base_model': BASE_MODEL,
    'base_model_revision': BASE_MODEL_REVISION,
    'legacy_adapter_used': False,
    'resume_checkpoint': resume_checkpoint,
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'best_metric': trainer.state.best_metric,
    'final_global_step': trainer.state.global_step,
    'baseline_metrics': baseline_metrics,
    'training_metrics': training_result.metrics,
    'final_metrics': final_metrics,
    'log_history': trainer.state.log_history,
    'dataset_hashes': actual_hashes,
    'token_lengths': length_report,
    'adapter_dir': str(ADAPTER_DIR),
}
(RUN_ROOT / 'training_report.json').write_text(json.dumps(training_report, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print(str(ADAPTER_DIR))

## 10. Validation üretim sağlık kontrolü

Bu kontrol holdout değildir. İlk beş validation kaydında greedy üretim yapar, JSON parse başarısını ve ham çıktıları rapora ekler.

In [ ]:
model.eval()
model.config.use_cache = True
validation_predictions = []

for row_index, row in enumerate(validation_rows[:5]):
    prompt_messages = row_messages(row)[:2]
    rendered_prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered_prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=1024, do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    completion_tokens = generated[0, inputs['input_ids'].shape[1]:]
    completion = tokenizer.decode(completion_tokens, skip_special_tokens=True).strip()
    try:
        parsed = json.loads(completion)
        json_valid = isinstance(parsed, dict)
    except json.JSONDecodeError:
        json_valid = False
    validation_predictions.append({'index': row_index, 'json_valid': json_valid, 'completion': completion, 'expected': row['output']})

(RUN_ROOT / 'validation_predictions.json').write_text(json.dumps(validation_predictions, ensure_ascii=False, indent=2), encoding='utf-8')
json_success_count = sum(item['json_valid'] for item in validation_predictions)
print(json.dumps({'checked': len(validation_predictions), 'valid_json': json_success_count, 'predictions_path': str(RUN_ROOT / 'validation_predictions.json')}, ensure_ascii=False, indent=2))

## 11. Teslim arşivi

Adaptör, tokenizer, sözleşme, eğitim raporu ve validation sağlık kontrolü tek ZIP arşivine alınır. Checkpoint dizini arşive dahil edilmez.

In [ ]:
DELIVERY_DIR = RUN_ROOT / 'delivery'
if DELIVERY_DIR.exists():
    shutil.rmtree(DELIVERY_DIR)
DELIVERY_DIR.mkdir(parents=True)
shutil.copytree(ADAPTER_DIR, DELIVERY_DIR / 'adapter_best')
for artifact_name in ['runtime_contract.json', 'training_report.json', 'validation_predictions.json']:
    shutil.copy2(RUN_ROOT / artifact_name, DELIVERY_DIR / artifact_name)
archive_path = shutil.make_archive(str(RUN_ROOT / f"{contract['run_name']}_delivery"), 'zip', DELIVERY_DIR)
print(archive_path)

## 12. Eğitim sonrası karar kapısı

ZIP arşivini ve `training_report.json` dosyasını projeye indirin. Yeni adaptörü önce donmuş OCR benchmark'ında eski base ve eski LoRA baseline ile karşılaştırın. Ardından daha önce görülmemiş bağımsız holdout setini yalnız bir kez değerlendirin. Precision, recall, F1, alan bazlı doğruluk, XSD geçerliliği ve deterministik tekrar koşulları birlikte geçmeden üretim adaptörünü değiştirmeyin.